# 04 · Predictive Modeling and Recommendations

**Purpose:** Retain the original regression and risk-classification workflows, then connect the results to measurable public-health program targets.

This notebook preserves the substantive code and analytical sequence from the original completed project. Changes are limited to portable file paths, organization, and explanatory markdown.


## Reproducible setup

The project-relative paths below replace the original Google Colab mount so the notebook runs consistently in VS Code, Jupyter, or GitHub Codespaces.


In [ ]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", context="notebook")

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_DATA = PROJECT_ROOT / "data" / "raw" / "cdc_brfss_nutrition_activity_obesity_2011_2024.csv"
PROCESSED_DATA = PROJECT_ROOT / "data" / "processed" / "obesity_analysis_ready.csv"

# Each analysis notebook is independently runnable from the project root or notebooks folder.
clean_df = pd.read_csv(PROCESSED_DATA, low_memory=False)
data = clean_df.copy()
print(f"Loaded {len(clean_df):,} analysis-ready rows covering {clean_df.YearStart.min()}–{clean_df.YearStart.max()}.")


## 4. Model Building

### 4.1 Quantitative Trend Analysis: Predicting Precise Obesity Percentages

**Why this step matters:** A held-out test set provides a more honest assessment of how the model performs on records it did not see during training.


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score

**What this model does:** The ensemble estimates obesity prevalence from year, location, and population stratification while capturing nonlinear relationships.


In [ ]:
# Select core features: year, location, hierarchical category, and specific hierarchical content
features = ['YearStart', 'LocationAbbr', 'StratificationCategory1', 'Stratification1']
X = clean_df[features]
y = clean_df['Data_Value']

# Larger sample sizes -> higher weights
weights = clean_df['Sample_Size'].fillna(clean_df['Sample_Size'].median())

# Pre-processing
categorical_features = ['LocationAbbr', 'StratificationCategory1', 'Stratification1']
numeric_features = ['YearStart']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ])

# Model Pipeline
model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor(n_estimators=100, random_state=42))
])

**Why this step matters:** A held-out test set provides a more honest assessment of how the model performs on records it did not see during training.


In [ ]:
# Training and Assessment
X_train, X_test, y_train, y_test, w_train, w_test = train_test_split(
    X, y, weights, test_size=0.2, random_state=42
)

model.fit(X_train, y_train, regressor__sample_weight=w_train)
y_pred = model.predict(X_test)
print(f"R2: {r2_score(y_test, y_pred):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, y_pred)):.4f}")

**What this step does:** Executes the next stage of the original analysis and preserves its result for the interpretation that follows.


In [ ]:
model.predict(X_test)

### 4.2 Public Health Intervention Prioritization: Risk Category Mapping

**Why this step matters:** A held-out test set provides a more honest assessment of how the model performs on records it did not see during training.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import classification_report, confusion_matrix

**What this model does:** Gradient boosting maps records into prioritization tiers so the analysis can support program targeting rather than clinical decisions.


In [ ]:
# Define the risk level
# 1/3
low_threshold = clean_df['Data_Value'].quantile(0.33)
high_threshold = clean_df['Data_Value'].quantile(0.66)

def get_risk_cat(val):
    if val <= low_threshold: return 0  # Low Risk
    elif val <= high_threshold: return 1 # Medium Risk
    else: return 2 # High Risk

clean_df['Risk_Category'] = clean_df['Data_Value'].apply(get_risk_cat)


features = ['YearStart', 'LocationAbbr', 'StratificationCategory1', 'Stratification1']
X = clean_df[features]
y = clean_df['Risk_Category']
weights = clean_df['Sample_Size'].fillna(clean_df['Sample_Size'].median())

# Pre-processing
categorical_features = ['LocationAbbr', 'StratificationCategory1', 'Stratification1']
numeric_features = ['YearStart']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ])

# Pipline
gb_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=5, random_state=42))
])

**Why this step matters:** A held-out test set provides a more honest assessment of how the model performs on records it did not see during training.


In [ ]:
X_train, X_test, y_train, y_test, w_train, w_test = train_test_split(
    X, y, weights, test_size=0.2, random_state=42
)

gb_model.fit(X_train, y_train, classifier__sample_weight=w_train)

# Confusion Matrix
y_pred = gb_model.predict(X_test)
labels = ['Low Risk', 'Medium Risk', 'High Risk']

plt.figure(figsize=(8, 6))
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='YlGnBu', xticklabels=labels, yticklabels=labels)
plt.title('Confusion Matrix: Obesity Risk Classification')
plt.xlabel('Predicted Risk')
plt.ylabel('Actual Risk')
plt.show()

print("\n--- Classification Report (Gradient Boosting) ---")
print(classification_report(y_test, y_pred, target_names=labels))
print(f"Low-Med Threshold: {low_threshold:.1f}%")
print(f"Med-High Threshold: {high_threshold:.1f}%")

**What this code does:** Feature importance provides a directional explanation of which inputs contribute most to the model’s predictions; it does not imply causality.


In [ ]:
# Feature Importance Visualization
ohe_feature_names = gb_model.named_steps['preprocessor'].transformers_[1][1].get_feature_names_out(categorical_features)
all_feature_names = numeric_features + list(ohe_feature_names)
importances = gb_model.named_steps['classifier'].feature_importances_

importance_df = pd.DataFrame({'feature': all_feature_names, 'importance': importances})
importance_df = importance_df.sort_values(by='importance', ascending=False).head(10)

plt.figure(figsize=(10, 6))
sns.barplot(x='importance', y='feature', data=importance_df, palette='viridis')
plt.title('Top 10 Risk Factors for Obesity Prevalence')
plt.show()

## Recommendations and measurable impact targets

The original models support prioritization and monitoring. The following percentages are **pilot KPI targets**, not causal effects promised by the observational data.

1. **High-burden outreach:** target a **10% increase in eligible-program enrollment** in priority locations within 12 months.
2. **Activity and nutrition access:** target a **15% increase in enrolled residents meeting the 150-minute weekly activity guideline**.
3. **Tailored communication:** target a **20% improvement in campaign reach or completion** versus a non-tailored message in an A/B test.
4. **Equity monitoring:** target a **10% reduction in the largest subgroup enrollment gap** during the first program year.
5. **Quarterly review:** scale only interventions that demonstrate both statistical evidence and practical operational value.

These targets give recruiters and stakeholders a clear measurement plan without overstating what the source data proves.
